# ⚽ Football Match Analysis - Demo Notebook

Ce notebook démontre les capacités du système d'analyse de matchs de football.

## 1. Setup et imports

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML
from pathlib import Path

# Nos modules
from config import config
from src.detector import PlayerBallDetector
from src.pose_estimator import PoseEstimator
from src.tracker import MultiObjectTracker
from src.team_classifier import TeamClassifier
from src.speed_calculator import SpeedDistanceCalculator
from src.heatmap_generator import HeatmapGenerator
from src.pitch_mapper import PitchMapper
from visualization.overlay import VideoOverlay
from utils.video_utils import get_video_info

print("✅ Modules chargés avec succès!")

## 2. Charger une vidéo de test

In [ ]:
# Chemin vers votre vidéo de test
VIDEO_PATH = "../input/match_sample.mp4"  # Modifiez ce chemin

# Vérifier si la vidéo existe
if Path(VIDEO_PATH).exists():
    info = get_video_info(VIDEO_PATH)
    print(f"📹 Vidéo: {VIDEO_PATH}")
    print(f"   Résolution: {info.width}x{info.height}")
    print(f"   FPS: {info.fps}")
    print(f"   Durée: {info.duration:.1f}s ({info.frame_count} frames)")
else:
    print(f"⚠️ Vidéo non trouvée: {VIDEO_PATH}")
    print("   Placez une vidéo dans le dossier input/ et modifiez VIDEO_PATH")

## 3. Détection de joueurs et ballon

In [ ]:
# Initialiser le détecteur
detector = PlayerBallDetector()
print("✅ Détecteur YOLO chargé")

# Charger un frame de test
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 100)  # Frame 100
ret, frame = cap.read()
cap.release()

if ret:
    # Détecter
    result = detector.detect(frame)
    
    print(f"\n📊 Résultats de détection:")
    print(f"   Joueurs détectés: {len(result.players)}")
    print(f"   Ballon détecté: {'Oui' if result.ball else 'Non'}")
    
    # Afficher les détections
    display_frame = frame.copy()
    
    for player in result.players:
        x1, y1, x2, y2 = map(int, player.bbox)
        cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(display_frame, f"{player.confidence:.2f}", 
                    (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    if result.ball:
        x1, y1, x2, y2 = map(int, result.ball.bbox)
        cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
    
    # Afficher
    plt.figure(figsize=(15, 10))
    plt.imshow(cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB))
    plt.title(f"Détection: {len(result.players)} joueurs, ballon: {'✓' if result.ball else '✗'}")
    plt.axis('off')
    plt.show()

## 4. Classification des équipes

In [ ]:
# Initialiser le classificateur
team_classifier = TeamClassifier()

# Calibrer avec quelques frames
cap = cv2.VideoCapture(VIDEO_PATH)
calibration_frames = []
calibration_detections = []

for i in range(30):
    ret, frame = cap.read()
    if ret:
        calibration_frames.append(frame)
        result = detector.detect(frame)
        calibration_detections.extend(result.players)

cap.release()

# Calibrer
if calibration_frames:
    mid_frame = calibration_frames[15]
    result = detector.detect(mid_frame)
    
    if team_classifier.calibrate(mid_frame, result.players):
        colors = team_classifier.get_team_colors()
        print("✅ Calibration réussie!")
        print("\nCouleurs détectées (BGR):")
        for team, color in colors.items():
            print(f"   {team}: {color}")
    else:
        print("⚠️ Calibration échouée - pas assez de joueurs détectés")

## 5. Tracking des joueurs

In [ ]:
# Initialiser le tracker
tracker = MultiObjectTracker()
speed_calc = SpeedDistanceCalculator(fps=info.fps)

# Traiter quelques frames
cap = cv2.VideoCapture(VIDEO_PATH)
results_history = []

for i in range(100):  # 100 frames
    ret, frame = cap.read()
    if not ret:
        break
    
    # Détecter
    detection = detector.detect(frame)
    
    # Tracker
    tracking = tracker.update(detection)
    
    # Classifier et calculer vitesse
    for player in tracking.players:
        player.team = team_classifier.classify_team(frame, player.bbox, player.track_id)
        player.speed = speed_calc.update(player.track_id, player.bottom_center)
    
    results_history.append(tracking)

cap.release()

# Afficher les stats
print(f"\n📊 Résultats du tracking sur 100 frames:")
print(f"   Tracks totaux: {tracker.get_statistics()['total_tracks']}")
print(f"   Tracks actifs: {tracker.get_statistics()['active_tracks']}")

# Afficher les vitesses
stats = speed_calc.get_summary_stats()
print(f"\n⚡ Statistiques de vitesse:")
print(f"   Vitesse moyenne: {stats['avg_current_speed']:.1f} km/h")
print(f"   Vitesse max atteinte: {stats['overall_max_speed']:.1f} km/h")
print(f"   Distance totale: {stats['total_distance']:.0f}m")

## 6. Génération de Heatmap

In [ ]:
# Initialiser le générateur de heatmap
heatmap_gen = HeatmapGenerator()
heatmap_gen.set_video_dimensions(info.width, info.height)

# Ajouter les positions depuis l'historique
for tracking in results_history:
    for player in tracking.players:
        heatmap_gen.add_position(player.track_id, player.bottom_center, player.team)

# Générer les heatmaps
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Heatmap globale
heatmap_global = heatmap_gen.generate_heatmap()
axes[0].imshow(heatmap_global.grid, cmap='jet')
axes[0].set_title('Heatmap Globale')
axes[0].axis('off')

# Heatmap équipe A
heatmap_a = heatmap_gen.generate_heatmap(team='team_a')
axes[1].imshow(heatmap_a.grid, cmap='Blues')
axes[1].set_title('Équipe A')
axes[1].axis('off')

# Heatmap équipe B
heatmap_b = heatmap_gen.generate_heatmap(team='team_b')
axes[2].imshow(heatmap_b.grid, cmap='Reds')
axes[2].set_title('Équipe B')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"\n📍 Statistiques de position:")
avg_pos = heatmap_gen.get_average_position()
if avg_pos:
    print(f"   Position moyenne globale: ({avg_pos[0]:.0f}, {avg_pos[1]:.0f})")
print(f"   Couverture du terrain: {heatmap_gen.get_coverage_percentage():.1f}%")

## 7. Visualisation du terrain 2D

In [ ]:
# Créer le pitch mapper
pitch_mapper = PitchMapper()

# Créer l'image du terrain
pitch_image = pitch_mapper.create_pitch_image()

# Afficher
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(pitch_image, cv2.COLOR_BGR2RGB))
plt.title('Terrain de Football 2D')
plt.axis('off')
plt.show()

print(f"\n📐 Dimensions du terrain:")
print(f"   Longueur: {config.pitch.length}m")
print(f"   Largeur: {config.pitch.width}m")
print(f"   Visualisation: {pitch_mapper.viz_width}x{pitch_mapper.viz_height}px")

## 8. Overlay complet sur un frame

In [ ]:
# Initialiser l'overlay
overlay = VideoOverlay()

# Charger un frame
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 50)
ret, frame = cap.read()
cap.release()

if ret and results_history:
    # Utiliser le dernier résultat de tracking
    tracking = results_history[-1]
    
    # Stats simulées
    stats = {
        'player_count': len(tracking.players),
        'possession_a': 55,
        'possession_b': 45,
        'max_speed': 28.5,
        'total_distance': 1250
    }
    
    # Dessiner l'overlay
    annotated = overlay.draw_all(
        frame,
        tracking.players,
        tracking.ball,
        poses=None,
        stats=stats,
        pitch_mapper=pitch_mapper,
        draw_trajectories=True,
        draw_minimap=True,
        draw_stats=True,
        draw_skeletons=False
    )
    
    # Afficher
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title('Frame avec Overlay Complet')
    plt.axis('off')
    plt.show()

## 9. Analyse complète d'une vidéo

In [ ]:
from main import FootballAnalyzer

# Créer l'analyseur
analyzer = FootballAnalyzer(
    enable_pose=False,  # Désactivé pour plus de rapidité
    enable_speed=True,
    enable_heatmap=True,
    enable_minimap=True,
    enable_stats=True
)

# Analyser (limité à 200 frames pour la démo)
OUTPUT_PATH = "../output/demo_analyzed.mp4"

print("🚀 Lancement de l'analyse...")
print("   (Limitée à 200 frames pour la démo)\n")

analyzer.analyze_video(
    VIDEO_PATH,
    OUTPUT_PATH,
    max_frames=200
)

print(f"\n✅ Analyse terminée!")
print(f"   Vidéo de sortie: {OUTPUT_PATH}")

## 10. Afficher la vidéo résultat

In [ ]:
# Afficher la vidéo dans le notebook
if Path(OUTPUT_PATH).exists():
    display(Video(OUTPUT_PATH, width=800))
else:
    print("⚠️ Vidéo de sortie non trouvée")

## 11. Charger et afficher les statistiques

In [ ]:
import json
import pandas as pd

STATS_PATH = "../output/match_stats.json"

if Path(STATS_PATH).exists():
    with open(STATS_PATH) as f:
        stats = json.load(f)
    
    print("📊 STATISTIQUES DU MATCH")
    print("=" * 40)
    print(f"\nDurée analysée: {stats['duration']:.1f}s")
    print(f"Frames traités: {stats['frames_analyzed']}")
    
    print(f"\n🔵 Équipe A:")
    print(f"   Possession: {stats['team_a']['possession']:.1f}%")
    print(f"   Distance: {stats['team_a']['total_distance']:.0f}m")
    print(f"   Sprints: {stats['team_a']['sprints']}")
    print(f"   Vitesse max: {stats['team_a']['max_speed']:.1f} km/h")
    
    print(f"\n🔴 Équipe B:")
    print(f"   Possession: {stats['team_b']['possession']:.1f}%")
    print(f"   Distance: {stats['team_b']['total_distance']:.0f}m")
    print(f"   Sprints: {stats['team_b']['sprints']}")
    print(f"   Vitesse max: {stats['team_b']['max_speed']:.1f} km/h")
    
    # Tableau des joueurs
    print(f"\n👥 Statistiques par joueur:")
    df = pd.DataFrame([
        {
            'ID': pid,
            'Équipe': data['team'],
            'Distance (m)': round(data['distance'], 1),
            'Vit. max (km/h)': round(data['max_speed'], 1),
            'Sprints': data['sprints']
        }
        for pid, data in stats['players'].items()
    ])
    display(df)
else:
    print("⚠️ Fichier de statistiques non trouvé")

---

## 🎉 Fin de la démo!

Vous avez vu les principales fonctionnalités du système:

1. ✅ Détection de joueurs et ballon avec YOLO
2. ✅ Classification automatique des équipes par couleur
3. ✅ Tracking multi-objets avec IDs persistants
4. ✅ Calcul des vitesses et distances
5. ✅ Génération de heatmaps
6. ✅ Visualisation du terrain 2D
7. ✅ Overlay professionnel style broadcast
8. ✅ Statistiques détaillées

Pour aller plus loin:
- Lancez le dashboard Streamlit: `streamlit run visualization/dashboard.py`
- Consultez le README pour toutes les options
- Activez l'estimation de pose pour les squelettes